In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader

from dataset_loaders import MPNNDataset

In [2]:
device = "cuda"

dataset = MPNNDataset(
    device,
    "graph_powerlaw_cluster_graph_n7",
    program="graph_coloring",
)

loader = DataLoader(dataset, batch_size=512, shuffle=True)

In [3]:
class ToyMPNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.lin_msg1 = nn.Linear(in_dim, hidden_dim)
        self.lin_msg2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, X, edge_index):
        """
        Forward pass for the MPNN with batching support.

        X: Tensor of shape (batch_size, num_nodes, in_dim)
        edge_index: Tensor of shape (2, num_edges) with the indices of the edges.
        batch_size: The number of graphs in the batch.
        """
        # Reshape X to have shape (batch_size * num_nodes, in_dim)
        batch_size = X.size(0)
        num_nodes = X.size(1)
        X_reshaped = X.view(-1, X.size(2))  # (batch_size * num_nodes, in_dim)

        # Get source and destination nodes from edge_index
        src, dst = edge_index
        num_edges = src.size(0)

        offsets = torch.arange(batch_size, device=X.device) * num_nodes  # (batch_size,)
        offsets = offsets.view(-1, 1).expand(-1, num_edges).reshape(-1)  # (batch_size * num_edges,)

        src = src.repeat(batch_size) + offsets
        dst = dst.repeat(batch_size) + offsets

        # Layer 1: Message Passing
        m1 = self.lin_msg1(X_reshaped[src])  # Message passing from src nodes
        agg1 = torch.zeros(
            X_reshaped.size(0), m1.size(1), device=X.device
        )  # Aggregation tensor
        agg1.index_add_(0, dst, m1)  # Aggregate messages at destination nodes
        h1 = F.relu(agg1)  # Apply ReLU activation

        # Layer 2: Message Passing
        m2 = self.lin_msg2(
            h1[src]
        )  # Message passing from src nodes (after first aggregation)
        agg2 = torch.zeros(
            h1.size(0), m2.size(1), device=X.device
        )  # Aggregation tensor
        agg2.index_add_(0, dst, m2)  # Aggregate messages at destination nodes
        h2 = F.relu(agg2)  # Apply ReLU activation

        return h2

In [4]:
# Hyperparameters
in_dim = 1
hidden_dim = 16
out_dim = in_dim  # number of classes
lr = 0.01
epochs = 50

# Model, loss, optimizer
model = ToyMPNN(in_dim, hidden_dim, out_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)

In [5]:
def print_grads(model):
    for name, param in model.named_parameters():
            if param.grad is not None:
                print(name, param.grad.abs().mean())

In [6]:
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    total_loss = 0

    for batch in loader:
        X = batch[0]
        y = batch[1].reshape(-1, batch[1].size(2))
        out = model(X, dataset.edge_index)  # forward pass
        loss = criterion(out, y)  # compute loss
        loss.backward()  # backward pass
        # print_grads(model)
        optimizer.step()  # update weights
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 239.1692
Epoch 2, Loss: 234.3730
Epoch 3, Loss: 234.4029
Epoch 4, Loss: 234.3310
Epoch 5, Loss: 234.3266
Epoch 6, Loss: 234.3498
Epoch 7, Loss: 234.3512
Epoch 8, Loss: 234.3666
Epoch 9, Loss: 234.2842
Epoch 10, Loss: 234.3724
Epoch 11, Loss: 234.2946
Epoch 12, Loss: 234.3531
Epoch 13, Loss: 234.3344
Epoch 14, Loss: 234.3139
Epoch 15, Loss: 234.3024
Epoch 16, Loss: 234.3562
Epoch 17, Loss: 234.3181
Epoch 18, Loss: 234.3499
Epoch 19, Loss: 234.3997
Epoch 20, Loss: 234.3376
Epoch 21, Loss: 234.3188
Epoch 22, Loss: 234.3629
Epoch 23, Loss: 234.3306
Epoch 24, Loss: 234.2886
Epoch 25, Loss: 234.3155
Epoch 26, Loss: 234.3393
Epoch 27, Loss: 234.3405
Epoch 28, Loss: 234.2831
Epoch 29, Loss: 234.3995
Epoch 30, Loss: 234.3134
Epoch 31, Loss: 234.3938
Epoch 32, Loss: 234.3252
Epoch 33, Loss: 234.3380
Epoch 34, Loss: 234.3796
Epoch 35, Loss: 234.3345
Epoch 36, Loss: 234.3575
Epoch 37, Loss: 234.3649
Epoch 38, Loss: 234.3906
Epoch 39, Loss: 234.3673
Epoch 40, Loss: 234.3590
Epoch 41,